<center><img src="img/banner.jpeg" width=1200px /></center>

<center><h1><b><span style="color: green">Translational Neuroscience School 2026</b></span></h1></center>

# <span style="color: red;">Taller de Análisis de Datos.</span>
## <span style="color: red;"> Psic. Rodrigo Cáceres Robles</span>
## Datos electrofisiológicos de: <span style="color: blue;">*Fear conditioning potentiates the hippocampal CA1 commissural pathway in vivo and increases awake phase sleep*</span>

---

### <span style="color: red;">Acerca de este taller</span>

Este taller está diseñado para aprender y practicar herramientas de Python utilizando registros electrofisiológicos reales obtenidos de ratones en libre movimiento. Los datos y análisis que exploraremos provienen del estudio publicado por Subramaniyan et al. (2021) en la revista *Hippocampus* (DOI: [10.1002/hipo.23381](https://doi.org/10.1002/hipo.23381)).

<center> <img src="img/hipo.v31.2.cover.jpg" width=200px /> </center>

Trabajar con datos reales nos permite enfrentarnos a los mismos problemas que encuentra un investigador: Ruido en las señales, variables de confusión, etc.

---

### <span style="color: red;">Contexto del estudio</span>

El hipocampo es una estructura cerebral esencial para el aprendizaje espacial y la formación de memorias. En este estudio, los autores investigaron si el aprendizaje asociativo produce cambios medibles en la fuerza sináptica de la vía comisural que conecta las regiones CA1 del hipocampo izquierdo y derecho.

Para ello, utilizaron un paradigma de condicionamiento contextual de miedo (cFC), en el que los ratones aprenden a asociar un contexto (la cámara de condicionamiento) con un estímulo aversivo (descargas eléctricas en las patas). Este tipo de aprendizaje ocurre en un solo ensayo de ~5 minutos, y como veremos provocará cambios medibles en la fuerza sináptica.

<center> <img src="img/articulo-e1.png" width=400px /> </center>
<span style="color: blue;"><center><b>Figura 1e </b> </center></span>

#### <span style="color: red;">¿Cómo midieron la fuerza sináptica?</span>

Los investigadores implantaron electrodos en el hipocampo dorsal de ratones:
- Un electrodo de estimulación en la región CA1 izquierda.
- Uno o más electrodos de registro en la región CA1 derecha.
<center> <img src="img/articulo-b1.png" width=400px /> </center>
<span style="color: blue;"><center><b>Figura 1b </b> </center></span>

La fuerza sináptica se cuantificó midiendo la pendiente de los potenciales postsinápticos excitatorios de campo (fEPSP), que refleja la eficacia de la transmisión sináptica en la vía comisural. Se aplicaron pulsos de estimulación cada 60 segundos durante 2 horas antes (línea base) y 3 horas después del condicionamiento.
<center> <img src="img/Radahmadi-2c.png" width=400px /> </center>
<span style="color: blue;"><center><b>Radahmadi et. al. (2025). Figura 2c. </b> </center></span>


#### <span style="color: red;">Principales hallazgos del estudio</span>

1. Después del condicionamiento de miedo, la fuerza sináptica de la vía comisural aumentó significativamente, lo que indica LTP in vivo asociado al aprendizaje.
2. Aunque los experimentos se realizaron durante la fase oscura (fase activa), los ratones durmieron más después del condicionamiento, con predominio de sueño NREM.
3. Durante el sueño NREM, las respuestas sinápticas fueron mayores que durante la vigilia, actuando como una modulación de ganancia multiplicativa
---

### <span style="color: red;">Objetivos del taller</span>

En este taller aprenderemos a:

1. Entender el flujo de trabajo para calcular una pendiente.
2. Cargar y explorar datos electrofisiológicos.
3. Visualizar series temporales de pendientes de fEPSP,
4. Manipular y realizar operaciones con arreglos numéricos.

---

### <span style="color: red;">Datos</span>

Los datos originales (en formato de MATLAB, .mat) asociados a este estudio están disponibles públicamente en el repositorio Dryad:

> **DOI:** [10.5061/dryad.sj3tx965f](https://doi.org/10.5061/dryad.sj3tx965f)

---

# Introducción visual: Calcular pendientes de sin(x) y luego fEPSP

## ¿Para qué sirve este notebook?

La derivada es parte de muchos flujos de trabajo en análisis de señales,

No necesitas saber programar para usar este notebook — solo mueve los controles y observa qué pasa.

### Objetivos específicos del notebook A
1. Ganar una intuición sobre qué es la primera derivada y qué nos dice sobre una curva
2. Qué es la segunda derivada y qué información adicional nos da
3. Qué significan los ceros, máximos y mínimos de cada derivada
5. Cómo aplicar todo esto para medir la pendiente de un fEPSP.

---

## 1. Preparación

Ejecuta esta celda para cargar las herramientas necesarias. No te preocupes por entender el código — lo importante son las gráficas.

In [2]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

---

## 2. ¿Qué es la derivada?

Imagina que caminas por una colina. La derivada nos dice qué tan empinado es el camino en cada punto:

- Si la derivada es positiva, estamos subiendo
- Si la derivada es negativa, estamos bajando
- Si la derivada es cero, estamos en un punto plano (una cima o un valle)
- Cuanto más grande (positiva o negativa), más empinado es el camino

Vamos a verlo con la función más simple posible: sin(x).

### Ejecuta la siguiente celda y mueve el control del punto rojo:

In [3]:
def plot_derivative_intro(x_pos):
    """Muestra sin(x), su tangente en un punto, y la primera derivada."""
    x = np.linspace(0, 4 * np.pi, 500)
    y = np.sin(x)
    dy = np.cos(x)  # derivada analítica de sin(x) = cos(x)
    
    # Valor en el punto seleccionado
    y_at_pos = np.sin(x_pos)
    dy_at_pos = np.cos(x_pos)
    
    # Línea tangente
    tangent_x = np.linspace(x_pos - 1.2, x_pos + 1.2, 50)
    tangent_y = y_at_pos + dy_at_pos * (tangent_x - x_pos)
    
    fig, axs = plt.subplots(2, 1, figsize=(7,7), sharex=True)
    
    # Panel superior: f(x) = sin(x)
    axs[0].plot(x, y, color='black', linewidth=1.5, label='f(x) = sin(x)')
    axs[0].plot(tangent_x, tangent_y, color='red', linewidth=2, alpha=0.8, label=f'Tangente (pendiente = {dy_at_pos:.2f})')
    axs[0].plot(x_pos, y_at_pos, 'ro', markersize=10, zorder=5)
    axs[0].axhline(0, color='black', alpha=0.2)
    axs[0].set_ylabel('f(x)')
    axs[0].set_title('Función original: sin(x)')
    axs[0].legend(fontsize=8, loc='upper right')
    axs[0].set_ylim(-1.8, 1.8)
    
    # Panel inferior: f'(x) = cos(x)
    axs[1].plot(x, dy, color='black', linewidth=1.5, label="f'(x) = cos(x)")
    axs[1].plot(x_pos, dy_at_pos, 'ro', markersize=10, zorder=5)
    axs[1].axhline(0, color='black', alpha=0.3, linewidth=1)
    axs[1].set_ylabel("f'(x)")
    axs[1].set_xlabel('x')
    axs[1].set_title('Primera derivada')
    axs[1].legend(fontsize=8, loc='upper right')
    axs[1].set_ylim(-1.8, 1.8)
    
    # Color del fondo según el signo de la derivada
    if dy_at_pos > 0.05:
        for ax in axs:
            ax.axvline(x_pos, color='green', alpha=0.15, linewidth=15)
        axs[0].annotate('SUBIENDO', xy=(x_pos, -1.5), fontsize=11, color='green',
                        ha='center', fontweight='bold')
    elif dy_at_pos < -0.05:
        for ax in axs:
            ax.axvline(x_pos, color='red', alpha=0.15, linewidth=15)
        axs[0].annotate('BAJANDO', xy=(x_pos, -1.5), fontsize=11, color='red',
                        ha='center', fontweight='bold')
    else:
        for ax in axs:
            ax.axvline(x_pos, color='blue', alpha=0.15, linewidth=15)
        axs[0].annotate('PLANO ―', xy=(x_pos, -1.5), fontsize=11, color='blue',
                        ha='center', fontweight='bold')
    
    fig.tight_layout()
    plt.show()

w1 = widgets.interactive(
    plot_derivative_intro,
    x_pos=widgets.FloatSlider(min=0.1, max=4*np.pi-0.1, step=0.05, value=1.0,
                               description='Posición x:', layout=widgets.Layout(width='80%'))
)
display(w1)

interactive(children=(FloatSlider(value=1.0, description='Posición x:', layout=Layout(width='80%'), max=12.466…

### ¿Qué observar?

Mueve el slider lentamente y fíjate en:

-  Cuando sin(x) sube:  la derivada (panel inferior) es positiva (por encima de cero)
-  Cuando sin(x) bajA: la derivada es negativa (por debajo de cero)
-  En las cimas y valles de sin(x): la derivada cruza por cero.
- Donde la pendiente es más empinada: la derivada tiene su valor máximo o mínimo.
- ES decir que laa primera derivada mide la velocidad de cambio. Su mínimo nos dice dónde la función baja más rápido.

---

## 3. La segunda derivada

La primera derivada nos dice la pendiente. Pero la segunda derivadaa nos dice cómo cambia la pendiente — es decir, la curvatura:

- Segunda derivada positiva: La curva es cóncava hacia arriba (forma de U) U
- Segunda derivada negativa: La curva es cóncava hacia abajo (forma de U invertida)
- Segunda derivada cero: punto de inflexión (la curvatura cambia de dirección)

### Ejecuta la celda y explora:

In [4]:
def plot_both_derivatives(x_pos):
    """Muestra f(x), f'(x), f''(x) y su significado geométrico."""
    x = np.linspace(0, 4 * np.pi, 500)
    y = np.sin(x)
    dy = np.cos(x)           # primera derivada
    d2y = -np.sin(x)         # segunda derivada
    
    y0 = np.sin(x_pos)
    dy0 = np.cos(x_pos)
    d2y0 = -np.sin(x_pos)
    
    # Tangente
    tx = np.linspace(x_pos - 1.0, x_pos + 1.0, 50)
    ty = y0 + dy0 * (tx - x_pos)
    
    fig, axs = plt.subplots(3, 1, figsize=(7, 7), sharex=True)
    
    # f(x) 
    axs[0].plot(x, y, 'black', linewidth=1.5)
    axs[0].plot(tx, ty, 'red', linewidth=2, alpha=0.7)
    axs[0].plot(x_pos, y0, 'ro', markersize=9, zorder=5)
    axs[0].axhline(0, color='black', alpha=0.2)
    axs[0].set_ylabel('f(x)')
    axs[0].set_title('f(x) = sin(x)')
    axs[0].set_ylim(-1.8, 1.8)
    
    # f'(x)
    axs[1].plot(x, dy, 'black', linewidth=1.5)
    axs[1].plot(x_pos, dy0, 'ro', markersize=9, zorder=5)
    axs[1].axhline(0, color='black', alpha=0.3)
    axs[1].set_ylabel("f'(x)")
    axs[1].set_title(f"Primera derivada: f'(x) = {dy0:.2f}")
    axs[1].set_ylim(-1.8, 1.8)
    
    # f''(x)
    axs[2].plot(x, d2y, 'black', linewidth=1.5)
    axs[2].plot(x_pos, d2y0, 'ro', markersize=9, zorder=5)
    axs[2].axhline(0, color='black', alpha=0.3)
    axs[2].set_ylabel("f''(x)")
    axs[2].set_xlabel('x')
    axs[2].set_title(f"Segunda derivada: f''(x) = {d2y0:.2f}")
    axs[2].set_ylim(-1.8, 1.8)
    
    # Anotaciones
    if d2y0 > 0.05:
        shape_text = 'Curvatura: ∪ (cóncava arriba)'
        color = 'green'
    elif d2y0 < -0.05:
        shape_text = 'Curvatura: ∩ (cóncava abajo)'
        color = 'red'
    else:
        shape_text = 'Punto de inflexión'
        color = 'blue'
    
    for ax in axs:
        ax.axvline(x_pos, color=color, alpha=0.12, linewidth=15)
    axs[2].annotate(shape_text, xy=(x_pos, -1.5), fontsize=10, color=color,
                    ha='center', fontweight='bold')
    
    fig.tight_layout()
    plt.show()

w2 = widgets.interactive(
    plot_both_derivatives,
    x_pos=widgets.FloatSlider(min=0.1, max=4*np.pi-0.1, step=0.05, value=np.pi/2,
                               description='Posición x:', layout=widgets.Layout(width='80%'))
)
display(w2)

interactive(children=(FloatSlider(value=1.5707963267948966, description='Posición x:', layout=Layout(width='80…

### Relación entre las tres curvas

Mueve el slider y observa estos momentos clave:

| Posición | f(x) | f'(x) | f''(x) |
|---|---|---|---|
| Cima de sin(x) | Máximo | Cruza por cero (de + a −) | Mínimo (más negativo) |
| Valle de sin(x) | Mínimo | Cruza por cero (de − a +) | Máximo (más positivo) |
| Punto de inflexión | Cruzando por cero | Máximo o mínimo | Cruza por cero |
| Máxima pendiente descendente | Bajando más rápido | Mínimo | Cruza por cero |

El mínimo de la primera derivada corresponde al punto de máxima pendiente descendente. Este es exactamente el punto que necesitamos para medir la pendiente del fEPSP.

---

## 5. Usando la derivada para medir pendientes

Ahora juntamos todo. Queremos medir la pendiente de la bajada de una curva similar al fEPSP.

El procedimiento es:
1. Calcular la primera derivada
2. Encontrar su mínimo (= punto de máxima bajada)
3. Centrar una ventana alrededor de ese punto
4. Hacer una regresión lineal (ajustar una recta) en esa ventana

Vamos a practicarlo con una función sintética que imita un fEPSP:

In [5]:
def simular_fepsp(x, amplitude=1.0, center=5.0, width=1.5):
    """Genera una curva similar a un fEPSP (deflexión negativa gaussiana)."""
    return -amplitude * np.exp(-((x - center) ** 2) / (2 * width ** 2))


def plot_slope_demo(amplitude, width, samp_window, use_d2):
    """Demo interactivo de cálculo de slope con señal tipo fEPSP."""
    np.random.seed(42)
    
    sr = 16000
    dt = 1.0 / sr
    t = np.arange(0, 0.015, dt)  # 15 ms
    t_ms = t * 1000
    
    # Señal tipo fEPSP
    signal = simular_fepsp(t_ms, amplitude=amplitude, center=7.0, width=width)
    
    # Derivada elegida
    d_order = 2 if use_d2 else 1
    dtrace = signal.copy()
    for _ in range(d_order):
        dtrace = np.gradient(dtrace, dt)
    
    # Encontrar pico
    if use_d2:
        peak_ix = np.argmin(dtrace)
    else:
        peak_ix = np.argmin(dtrace)
    
    # Ventana y regresión
    left_ix = max(0, peak_ix - samp_window)
    right_ix = min(len(signal), peak_ix + samp_window)
    
    x_fit = np.arange(left_ix, right_ix)
    y_fit = signal[left_ix:right_ix]
    
    slope_samp, intercept = np.polyfit(x_fit, y_fit, 1)
    slope_sec = slope_samp * sr
    fit_line = slope_samp * x_fit + intercept
    
    # Línea extendida
    x_ext = np.arange(max(0, left_ix - 30), min(len(signal), right_ix + 30))
    fit_ext = slope_samp * x_ext + intercept
    
    
    fig, axs = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
    
    # Señal + ajuste
    axs[0].plot(t_ms, signal, color='black', linewidth=0.8, label='Señal')
    axs[0].plot(t_ms[x_fit], fit_line, color='red', linewidth=2.5,
                label=f'Slope = {slope_sec:.1f} V/s')
    axs[0].plot(t_ms[x_ext], fit_ext, color='red', linewidth=1, linestyle='dashed', alpha=0.4)
    axs[0].axvspan(t_ms[left_ix], t_ms[right_ix], alpha=0.08, color='red')
    axs[0].axvline(t_ms[peak_ix], color='blue', linewidth=0.7, alpha=0.6)
    axs[0].set_ylabel('Voltaje (V)')
    method_name = 'Segunda derivada (d2)' if use_d2 else 'Primera derivada (d1)'
    axs[0].set_title(f'Método: {method_name} | Ventana: ±{samp_window} muestras')
    axs[0].legend(fontsize=7, loc='lower right')
    
    # Derivada
    axs[1].plot(t_ms, dtrace, color='black', linewidth=0.8)
    axs[1].axvline(t_ms[peak_ix], color='blue', linewidth=0.7, alpha=0.6, label=f'Pico d{d_order}')
    axs[1].axhline(0, color='black', alpha=0.2)
    axs[1].set_ylabel(f'd{d_order}V/dt')
    axs[1].set_xlabel('Tiempo (ms)')
    axs[1].legend(fontsize=8)
    
    fig.tight_layout()
    plt.show()

w4 = widgets.interactive(
    plot_slope_demo,
    amplitude=widgets.FloatSlider(min=0.2, max=2.0, step=0.1, value=1.0,
                                   description='Amplitud:'),
    width=widgets.FloatSlider(min=0.5, max=3.0, step=0.1, value=1.5,
                               description='Ancho:'),
    samp_window=widgets.IntSlider(min=3, max=60, value=10,
                                   description='Ventana (±n):'),
    use_d2=widgets.Checkbox(value=False, description='Usar d2 (en vez de d1)')
)
display(w4)


interactive(children=(FloatSlider(value=1.0, description='Amplitud:', max=2.0, min=0.2), FloatSlider(value=1.5…

---


<p>
  <img src="img/cat_dancing.gif"
       width="60"
       height="60"
       style="border-radius: 50%; object-fit: cover; border: 1px solid blue;" />
  <span style="color: blue; font-size: 40px;">Ejercicios</span>
</p>

**Ejercicio 1: Efecto del tamaño de ventana**
1. Prueba ventanas de 5, 10, 20, 40.
2. Observa cómo la línea roja punteada se aleja de la curva cuando la ventana es demasiado grande.

**Ejercicio 2: Señales con distinta forma**
1. Cambia la amplitud y el ancho. ¿Cómo afecta al slope calculado?

---